In [2]:
import importlib
import sys

# ──────────── Paths (update for your machine) ────────────
ijepa_checkpoint = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/scripts/SEG-RDM/rdm/pretrained_enc_ckpts/ijepa/IN1K-vit.h.14-300e.pth.tar"
repo_root = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/scripts/StyleGAN2/seg-aware-stylegan2"
data_path = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/dataset/256/imagenet.zip"
out_base = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/scripts/StyleGAN2/seg-aware-stylegan2/outputs/debug_sam"

# SAM on-the-fly extraction (fallback for missing pre-computed)
sam_checkpoint = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/scripts/segProto/checkpoints/sam_vit_b_01ec64.pth"
# sam_cache_dir = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/sam_cache_unified"  # same dir as pre-computed!
sam_cache_dir = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/sam_temp"

# Pre-computed embeddings
# sam_npz_dir = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/sam_cache_unified"
sam_npz_dir ="/scratch/gilbreth/abelde/Thesis/StructureAwareGen/sam_temp"
ijepa_npz_dir = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/ijepa_embeddings"
origin_map_json = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/dataset/256/origin_map_imagenetdebug.json"

# RDM mixed training (set to a checkpoint path to enable, or None to skip)
rdm_checkpoint = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/scripts/SEG-RDM/rdm/rdm_out_final/4_nodes/batch_64/ijepa_and_seg_aware/checkpoint-last.pth"

# ──────────── Hyperparams ────────────
SAM_PROB = 0.5            # Network sees SAM tokens 50% of the time
SEM_MIX = 0.9
FUSION_ALPHA = 0.2
LAMBDA_SEG_ALIGN = 0.1
LAMBDA_SEG_DIVERSITY = 0.05

outdir = f"{out_base}/debug-run"

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import train
importlib.reload(train)

# ──────────── Build CLI args ────────────
sys.argv = [
    "train.py",
    "--outdir", outdir,
    "--data", data_path,
    "--subset", "20",       # LIMIT TO 20 IMAGES FOR DEBUGGING
    "--gpus", "1",
    "--cond", "1",
    "--batch", "4",
    "--kimg", "1",

    # I-JEPA options
    "--ijepa_checkpoint", ijepa_checkpoint,
    "--ijepa_lambda", "1.0",
    "--ijepa_image", "256",
    "--ijepa_input_channel", "3",
    "--ijepa_dim", "1280",
    "--ijepa_warmup_kimg", "0.5",
    "--sem_mixing_prob", str(SEM_MIX),
    "--fusion_alpha", str(FUSION_ALPHA),
    "--origin-map-json", origin_map_json,

    # Pre-computed embeddings (AlignedSegDataset loads from npz)
    "--use-seg-embeddings",
    "--sam-npz-dir", sam_npz_dir,
    "--ijepa-npz-dir", ijepa_npz_dir,

    # SAM on-the-fly fallback (extracts missing, saves to sam_cache_dir)
    "--sam-enabled", "true",
    "--sam-prob", str(SAM_PROB),
    "--sam-checkpoint", sam_checkpoint,
    "--sam-cache-dir", sam_cache_dir,
    "--sam-model-type", "vit_b",
    "--sam-max-masks", "250",
    "--sam-emb-logging", "true",

    # Alignment & diversity loss weights
    "--lambda-seg-align", str(LAMBDA_SEG_ALIGN),
    "--lambda-seg-diversity", str(LAMBDA_SEG_DIVERSITY),

    "--resume", "noresume",
]

# ── Optional: RDM mixed training ──
if rdm_checkpoint is not None:
    sys.argv += [
        "--rdm-checkpoint", rdm_checkpoint,
        "--rdm-mix-prob", "0.3",
        "--rdm-warmup-kimg", "0",
    ]

# Run

train.main(standalone_mode=False)

Using AlignedSegDataset with pre-computed embeddings
  SAM embeddings: /scratch/gilbreth/abelde/Thesis/StructureAwareGen/sam_temp
  I-JEPA embeddings: /scratch/gilbreth/abelde/Thesis/StructureAwareGen/ijepa_embeddings
  Origin map: /scratch/gilbreth/abelde/Thesis/StructureAwareGen/dataset/256/origin_map_imagenetdebug.json
  Max segments: 250
  Loaded origin_map with 5000 entries from /scratch/gilbreth/abelde/Thesis/StructureAwareGen/dataset/256/origin_map_imagenetdebug.json
  origin_map sanity check: '00000/img00000000.png' -> '0/105557' OK
AlignedSegDataset initialized:
  Images: /scratch/gilbreth/abelde/Thesis/StructureAwareGen/dataset/256/imagenet.zip
  SAM embeddings: /scratch/gilbreth/abelde/Thesis/StructureAwareGen/sam_temp
  I-JEPA embeddings: /scratch/gilbreth/abelde/Thesis/StructureAwareGen/ijepa_embeddings
  Origin map entries: 5000
  Max segments: 250
  Use labels (conditional): True
[3, 256, 256]

Training options:
{
  "num_gpus": 1,
  "image_snapshot_ticks": 50,
  "network

: 

In [ ]:
import zipfile
z=zipfile.ZipFile('/scratch/gilbreth/abelde/Thesis/StructureAwareGen/dataset/256/imagenet_debug_subset.zip')
print(z.namelist()[:20])

In [ ]:
cd /scratch/gilbreth/abelde/Thesis/StructureAwareGen/scripts/StyleGAN2/seg-aware-stylegan2
python generate_origin_map.py \
    --source /scratch/gilbreth/abelde/Thesis/StructureAwareGen/dataset/imagenet_debug_subset/train \
    --dest /scratch/gilbreth/abelde/Thesis/StructureAwareGen/dataset/256/origin_map_imagenetdebug.json

In [1]:
import os
import numpy as np

npz_path = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/ijepa_embeddings/1/2470.npz"

print("File:", npz_path)
print("On-disk size (MB):", os.path.getsize(npz_path) / (1024**2))

data = np.load(npz_path, allow_pickle=True)

print("\nKeys:", data.files)

for k in data.files:
    v = data[k]
    if isinstance(v, np.ndarray):
        print(f"\n{k}:")
        print("  shape:", v.shape)
        print("  dtype:", v.dtype)
        print("  nbytes in RAM (MB):", v.nbytes / (1024**2))
        # quick stats (skip for object arrays)
        if v.dtype != object and v.size > 0:
            try:
                print("  min/max:", float(np.min(v)), float(np.max(v)))
                print("  mean/std:", float(np.mean(v)), float(np.std(v)))
            except Exception:
                pass
    else:
        print(f"\n{k}: (non-ndarray) type={type(v)}")

data.close()


File: /scratch/gilbreth/abelde/Thesis/StructureAwareGen/ijepa_embeddings/1/2470.npz
On-disk size (MB): 0.0047588348388671875

Keys: ['emb']

emb:
  shape: (1280,)
  dtype: float32
  nbytes in RAM (MB): 0.0048828125
  min/max: -3.570686101913452 2.9386324882507324
  mean/std: -7.450580596923828e-09 0.9999998807907104


In [2]:
import os
import torch

pth_path = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/scripts/SEG-RDM/rdm/rdm_out_final/4_nodes/batch_64/ijepa_and_seg_aware/checkpoint-last.pth"

print("File:", pth_path)
print("On-disk size (MB):", os.path.getsize(pth_path) / (1024**2))

ckpt = torch.load(pth_path, map_location="cpu", weights_only=False)

def summarize_obj(obj, indent=0, max_items=50):
    pad = " " * indent
    if isinstance(obj, dict):
        keys = list(obj.keys())
        print(f"{pad}dict with {len(keys)} keys")
        for k in keys[:max_items]:
            v = obj[k]
            print(f"{pad}- {k!r}: {type(v).__name__}", end="")
            # common cases
            if torch.is_tensor(v):
                print(f" | shape={tuple(v.shape)} dtype={v.dtype} device={v.device}")
            elif isinstance(v, (list, tuple)):
                print(f" | len={len(v)}")
            elif isinstance(v, dict):
                print(" | dict")
            else:
                # print small scalars/strings
                if isinstance(v, (int, float, str, bool)):
                    print(f" | value={v!r}")
                else:
                    print()
        if len(keys) > max_items:
            print(f"{pad}... (showing first {max_items} keys)")
    elif torch.is_tensor(obj):
        print(f"{pad}Tensor shape={tuple(obj.shape)} dtype={obj.dtype} device={obj.device}")
    elif isinstance(obj, (list, tuple)):
        print(f"{pad}{type(obj).__name__} len={len(obj)}")
        for i, v in enumerate(obj[:min(len(obj), 10)]):
            print(f"{pad}- [{i}] {type(v).__name__}")
    else:
        print(f"{pad}{type(obj).__name__}: {obj!r}")

print("\nTop-level object:")
summarize_obj(ckpt, indent=2)


File: /scratch/gilbreth/abelde/Thesis/StructureAwareGen/scripts/SEG-RDM/rdm/rdm_out_final/4_nodes/batch_64/ijepa_and_seg_aware/checkpoint-last.pth
On-disk size (MB): 3711.553391456604

Top-level object:
  dict with 7 keys
  - 'model': OrderedDict | dict
  - 'model_ema': OrderedDict | dict
  - 'optimizer': dict | dict
  - 'epoch': int | value=28
  - 'scaler': dict | dict
  - 'args': Namespace
  - 'config': dict | dict


In [2]:
import torch

pth_path = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/scripts/SEG-RDM/rdm/rdm_out_final/4_nodes/batch_64/ijepa_and_seg_aware/checkpoint-last.pth"
ckpt = torch.load(pth_path, map_location="cpu", weights_only = False)
print("hello")
# Find the state_dict (common patterns)
state_dict = None
if isinstance(ckpt, dict):
    if "state_dict" in ckpt and isinstance(ckpt["state_dict"], dict):
        state_dict = ckpt["state_dict"]
    elif "model" in ckpt and isinstance(ckpt["model"], dict):
        state_dict = ckpt["model"]
    elif "G" in ckpt and isinstance(ckpt["G"], dict):
        state_dict = ckpt["G"]
    elif all(isinstance(v, torch.Tensor) for v in ckpt.values()):
        # checkpoint itself is a state_dict
        state_dict = ckpt

if state_dict is None:
    print("No obvious state_dict found. Top-level keys:", list(ckpt.keys()) if isinstance(ckpt, dict) else type(ckpt))
else:
    total_params = 0
    total_bytes = 0

    print("state_dict keys:", len(state_dict))
    for k, v in list(state_dict.items())[:200]:  # adjust limit
        if torch.is_tensor(v):
            n = v.numel()
            total_params += n
            total_bytes += v.element_size() * n
            print(f"{k:80s}  shape={tuple(v.shape)}  dtype={v.dtype}")
        else:
            print(f"{k:80s}  (non-tensor) type={type(v).__name__}")

    print("\nTotals:")
    print("  params:", total_params)
    print("  approx size in RAM (MB):", total_bytes / (1024**2))


hello
state_dict keys: 638
betas                                                                             shape=(1000,)  dtype=torch.float32
alphas_cumprod                                                                    shape=(1000,)  dtype=torch.float32
alphas_cumprod_prev                                                               shape=(1000,)  dtype=torch.float32
sqrt_alphas_cumprod                                                               shape=(1000,)  dtype=torch.float32
sqrt_one_minus_alphas_cumprod                                                     shape=(1000,)  dtype=torch.float32
log_one_minus_alphas_cumprod                                                      shape=(1000,)  dtype=torch.float32
sqrt_recip_alphas_cumprod                                                         shape=(1000,)  dtype=torch.float32
sqrt_recipm1_alphas_cumprod                                                       shape=(1000,)  dtype=torch.float32
posterior_variance                   

In [ ]:
!tensorboard --logdir /scratch/gilbreth/abelde/Thesis/StructureAwareGen/scripts/SEG-RDM/rdm/rdm_out_final/IJEPA_local_feat/a100-rerun/2_nodes --port 0


/home/abelde/.local/lib/python3.9/site-packages/tensorboard/default.py:30: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2026-02-27 22:13:21.909267: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772248402.306695  127553 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772248402.409707  127553 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772248403.375772  127553 computation_placer.cc:177] computation placer already regis

In [4]:
import torch
from omegaconf import OmegaConf

rdm_checkpoint = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/scripts/SEG-RDM/rdm/rdm_out/4_nodes/batch_128/ijepa_h14/checkpoint-last.pth"

ckpt = torch.load(rdm_checkpoint, map_location='cpu', weights_only=False)

print("=" * 80)
print("CHECKPOINT KEYS:")
print("=" * 80)
print(ckpt.keys())
print()

print("=" * 80)
print("CONFIG STRUCTURE:")
print("=" * 80)
config = ckpt.get('config')
if config is not None:
    # Print as YAML for readability
    if isinstance(config, dict):
        print(OmegaConf.to_yaml(OmegaConf.create(config)))
    else:
        print(OmegaConf.to_yaml(config))
else:
    print("No 'config' key found!")
print()

print("=" * 80)
print("CONFIG['MODEL'] KEYS:")
print("=" * 80)
if config and 'model' in config:
    print(config['model'].keys() if hasattr(config['model'], 'keys') else dir(config['model']))
    print()
    print("CONFIG['MODEL']['PARAMS'] KEYS:")
    print("-" * 40)
    if 'params' in config['model']:
        params = config['model']['params']
        print(params.keys() if hasattr(params, 'keys') else dir(params))
        print()
        if 'pretrained_enc_config' in params:
            print("CONFIG['MODEL']['PARAMS']['PRETRAINED_ENC_CONFIG']:")
            print("-" * 40)
            print(params['pretrained_enc_config'])
else:
    print("No 'model' key in config!")

CHECKPOINT KEYS:
dict_keys(['model', 'model_ema', 'optimizer', 'epoch', 'scaler', 'args'])

CONFIG STRUCTURE:
No 'config' key found!

CONFIG['MODEL'] KEYS:
No 'model' key in config!


In [1]:
import numpy as np
data = np.load('/scratch/gilbreth/abelde/Thesis/StructureAwareGen/sam_cache_unified/1/masks_npz/5129.npz')
emb = data['emb']
print(f'Shape: {emb.shape}')
print(f'Mean: {emb.mean():.3f}, Std: {emb.std():.3f}')
# Check if segments are too similar
from sklearn.metrics.pairwise import cosine_similarity
sim = cosine_similarity(emb)
print(f'Avg pairwise similarity: {sim.mean():.3f}')

Shape: (21, 256)
Mean: 0.012, Std: 0.134
Avg pairwise similarity: 0.721


In [3]:
import numpy as np
data = np.load('/scratch/gilbreth/abelde/Thesis/StructureAwareGen/sam_cache_unified/1/masks_npz/24443.npz')
emb = data['emb']
print(f'Shape: {emb.shape}')
print(f'Mean: {emb.mean():.3f}, Std: {emb.std():.3f}')
# Check if segments are too similar
from sklearn.metrics.pairwise import cosine_similarity
sim = cosine_similarity(emb)
print(f'Avg pairwise similarity: {sim.mean():.3f}')

Shape: (100, 256)
Mean: 0.014, Std: 0.132
Avg pairwise similarity: 0.701


In [10]:
import numpy as np
import glob
from sklearn.metrics.pairwise import cosine_similarity
from pathlib import Path

# Analyze first 5 classes (subfolders 0, 1, 2, 3, 4)
for class_id in range(5):
    print(f"\n{'='*60}")
    print(f"CLASS {class_id}")
    print(f"{'='*60}")
    
    similarities = []
    zero_variance_count = 0
    total_files = 0
    
    pattern = f'/scratch/gilbreth/abelde/Thesis/StructureAwareGen/sam_cache_unified/{class_id}/masks_npz/*.npz'
    npz_files = glob.glob(pattern)
    
    if len(npz_files) == 0:
        print(f"No files found for class {class_id}")
        continue
    
    for npz_file in npz_files:
        total_files += 1
        data = np.load(npz_file)
        emb = data['emb']
        
        if emb.shape[0] > 1:
            # Check for zero variance embeddings
            std_devs = emb.std(axis=1)
            if np.any(std_devs == 0):
                zero_variance_count += 1
                continue  # Skip files with constant embeddings
            
            # Use cosine similarity instead of correlation
            sim_matrix = cosine_similarity(emb)
            # Get upper triangle (excluding diagonal)
            triu_indices = np.triu_indices_from(sim_matrix, k=1)
            similarities.append(sim_matrix[triu_indices].mean())
    
    if len(similarities) == 0:
        print(f"No valid embeddings found for class {class_id}")
        continue
    
    print(f"Total files: {total_files}")
    print(f"Zero-variance files: {zero_variance_count} ({100*zero_variance_count/total_files:.1f}%)")
    print(f"Valid files: {len(similarities)}")
    print(f"\nAverage cosine similarity: {np.mean(similarities):.3f}")
    print(f"Std dev: {np.std(similarities):.3f}")
    print(f"Min: {np.min(similarities):.3f}, Max: {np.max(similarities):.3f}")
    
    # Distribution analysis
    sims = np.array(similarities)
    print(f"\nSimilarity distribution:")
    print(f"  < 0.4 (good diversity): {100*np.sum(sims < 0.4)/len(sims):.1f}%")
    print(f"  0.4-0.6 (moderate):     {100*np.sum((sims >= 0.4) & (sims < 0.6))/len(sims):.1f}%")
    print(f"  0.6-0.8 (high sim):     {100*np.sum((sims >= 0.6) & (sims < 0.8))/len(sims):.1f}%")
    print(f"  > 0.8 (collapsed):      {100*np.sum(sims >= 0.8)/len(sims):.1f}%")
    
    # Show a few example files with their similarities
    print(f"\nSample files:")
    sample_indices = np.random.choice(len(npz_files), min(3, len(npz_files)), replace=False)
    for idx in sample_indices:
        file_path = npz_files[idx]
        file_name = Path(file_path).name
        if idx < len(similarities):
            print(f"  {file_name}: similarity = {similarities[idx]:.3f}")

print(f"\n{'='*60}")
print("SUMMARY ACROSS FIRST 5 CLASSES")
print(f"{'='*60}")


CLASS 0
Total files: 828
Zero-variance files: 27 (3.3%)
Valid files: 801

Average cosine similarity: 0.701
Std dev: 0.044
Min: 0.577, Max: 0.926

Similarity distribution:
  < 0.4 (good diversity): 0.0%
  0.4-0.6 (moderate):     0.1%
  0.6-0.8 (high sim):     96.8%
  > 0.8 (collapsed):      3.1%

Sample files:
  577045.npz: similarity = 0.664
  1277622.npz: similarity = 0.650
  263616.npz: similarity = 0.652

CLASS 1
Total files: 833
Zero-variance files: 2 (0.2%)
Valid files: 830

Average cosine similarity: 0.708
Std dev: 0.050
Min: 0.589, Max: 0.980

Similarity distribution:
  < 0.4 (good diversity): 0.0%
  0.4-0.6 (moderate):     0.2%
  0.6-0.8 (high sim):     94.1%
  > 0.8 (collapsed):      5.7%

Sample files:
  1069060.npz: similarity = 0.686
  582644.npz: similarity = 0.671
  525265.npz: similarity = 0.691

CLASS 2
Total files: 520
Zero-variance files: 0 (0.0%)
Valid files: 520

Average cosine similarity: 0.726
Std dev: 0.056
Min: 0.633, Max: 0.926

Similarity distribution:
  < 0.

In [1]:
import tarfile

path = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/ijepa_embeddings.tar"  # or .tar.gz / .tgz
with tarfile.open(path, "r:*") as tf:   # r:* auto-detects compression
    names = tf.getnames()
    print("num files:", len(names))
    print("\n".join(names[:50]))  # first 50

num files: 1282168
scratch/gilbreth/abelde/Thesis/StructureAwareGen/ijepa_embeddings
scratch/gilbreth/abelde/Thesis/StructureAwareGen/ijepa_embeddings/891
scratch/gilbreth/abelde/Thesis/StructureAwareGen/ijepa_embeddings/891/530707.npz
scratch/gilbreth/abelde/Thesis/StructureAwareGen/ijepa_embeddings/891/748598.npz
scratch/gilbreth/abelde/Thesis/StructureAwareGen/ijepa_embeddings/891/892891.npz
scratch/gilbreth/abelde/Thesis/StructureAwareGen/ijepa_embeddings/891/916502.npz
scratch/gilbreth/abelde/Thesis/StructureAwareGen/ijepa_embeddings/891/587338.npz
scratch/gilbreth/abelde/Thesis/StructureAwareGen/ijepa_embeddings/891/1255713.npz
scratch/gilbreth/abelde/Thesis/StructureAwareGen/ijepa_embeddings/891/937484.npz
scratch/gilbreth/abelde/Thesis/StructureAwareGen/ijepa_embeddings/891/684852.npz
scratch/gilbreth/abelde/Thesis/StructureAwareGen/ijepa_embeddings/891/506357.npz
scratch/gilbreth/abelde/Thesis/StructureAwareGen/ijepa_embeddings/891/877703.npz
scratch/gilbreth/abelde/Thesis/Str

In [2]:
# Convert all ijepa npz files into one HDF5
import h5py
import numpy as np
import tarfile
from io import BytesIO
from tqdm import tqdm

tar_path = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/ijepa_embeddings.tar"
hdf5_path = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/ijepa_embeddings.h5"

with tarfile.open(tar_path, "r:") as tf, h5py.File(hdf5_path, "w") as hf:
    members = [m for m in tf.getmembers() if m.name.endswith(".npz")]
    print(f"Converting {len(members)} files...")
    for m in tqdm(members):
        f = tf.extractfile(m)
        data = np.load(BytesIO(f.read()), allow_pickle=True)
        # Use the path as the key e.g. "1/2470"
        key = m.name.replace(".npz", "").lstrip("/")
        grp = hf.require_group(key)
        for k in data.files:
            grp.create_dataset(k, data=data[k])

print("Done!")

Converting 1281167 files...


100%|██████████| 1281167/1281167 [16:41<00:00, 1279.40it/s]


Done!


In [2]:
# Convert all npz files from a folder into one HDF5
import h5py
import numpy as np
from pathlib import Path
from tqdm import tqdm
import warnings

folder_path = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/sam_cache_unified"
hdf5_path = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/sam_cache_unified.h5"

# Collect all npz files first
npz_files = list(Path(folder_path).rglob("*.npz"))
print(f"Converting {len(npz_files)} files...")

skipped_datasets = []
errors = []

with h5py.File(hdf5_path, "w") as hf:
    for fpath in tqdm(npz_files):
        try:
            data = np.load(fpath, allow_pickle=True)
            # Key e.g. "1/masks_npz/2470" (relative to folder_path)
            key = str(fpath.relative_to(folder_path)).replace(".npz", "")
            grp = hf.require_group(key)
            
            for k in data.files:
                arr = data[k]
                
                # Skip object dtype arrays
                if arr.dtype == np.object_:
                    skipped_datasets.append(f"{key}/{k}")
                    continue
                
                # Handle special dtypes
                if arr.dtype.kind == 'U':  # Unicode strings
                    # Convert to fixed-length bytes
                    arr = arr.astype('S')
                
                try:
                    grp.create_dataset(k, data=arr, compression="gzip", compression_opts=4)
                except (TypeError, ValueError) as e:
                    # Fallback: try without compression
                    try:
                        grp.create_dataset(k, data=arr)
                    except Exception as e2:
                        skipped_datasets.append(f"{key}/{k}: {str(e2)}")
            
            data.close()
            
        except Exception as e:
            errors.append(f"{fpath}: {str(e)}")

print("\nDone!")
print(f"Skipped {len(skipped_datasets)} datasets with incompatible dtypes")
if len(skipped_datasets) > 0:
    print("First 10 skipped:", skipped_datasets[:10])
if len(errors) > 0:
    print(f"\nEncountered {len(errors)} file-level errors:")
    print(errors[:10])

Converting 514636 files...


  0%|          | 11/514636 [00:00<10:03:23, 14.21it/s]

 48%|████▊     | 248824/514636 [5:01:52<5:22:28, 13.74it/s]   


KeyboardInterrupt: 

In [1]:
# Convert all npz files from a folder into one HDF5
import h5py
import numpy as np
from pathlib import Path
from tqdm import tqdm
import warnings

folder_path = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/region_emb_extract-a100-0.65dedup"
hdf5_path = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/region_emb_extract-a100-0.65dedup.h5"

# Collect all npz files first
npz_files = list(Path(folder_path).rglob("*.npz"))
print(f"Converting {len(npz_files)} files...")

skipped_datasets = []
errors = []

with h5py.File(hdf5_path, "w") as hf:
    for fpath in tqdm(npz_files):
        try:
            data = np.load(fpath, allow_pickle=True)
            # Key e.g. "1/masks_npz/2470" (relative to folder_path)
            key = str(fpath.relative_to(folder_path)).replace(".npz", "")
            grp = hf.require_group(key)
            
            for k in data.files:
                arr = data[k]
                
                # Skip object dtype arrays
                if arr.dtype == np.object_:
                    skipped_datasets.append(f"{key}/{k}")
                    continue
                
                # Handle special dtypes
                if arr.dtype.kind == 'U':  # Unicode strings
                    # Convert to fixed-length bytes
                    arr = arr.astype('S')
                
                try:
                    grp.create_dataset(k, data=arr, compression="gzip", compression_opts=4)
                except (TypeError, ValueError) as e:
                    # Fallback: try without compression
                    try:
                        grp.create_dataset(k, data=arr)
                    except Exception as e2:
                        skipped_datasets.append(f"{key}/{k}: {str(e2)}")
            
            data.close()
            
        except Exception as e:
            errors.append(f"{fpath}: {str(e)}")

print("\nDone!")
print(f"Skipped {len(skipped_datasets)} datasets with incompatible dtypes")
if len(skipped_datasets) > 0:
    print("First 10 skipped:", skipped_datasets[:10])
if len(errors) > 0:
    print(f"\nEncountered {len(errors)} file-level errors:")
    print(errors[:10])

Converting 510172 files...


100%|██████████| 510172/510172 [8:29:29<00:00, 16.69it/s]    



Done!
Skipped 0 datasets with incompatible dtypes


In [8]:
# ============================================================
# Convert raw NPZ folder → efficient flat H5
# Source: region_emb_extract-a100-0.65dedup/{class}/masks_npz/*.npz
# Each NPZ has: emb (N,256) float32, scores (N,) float32, shape (3,) int32
# Drops: label_map (redundant), packed (not used in training), meta/
# ============================================================
import os, numpy as np, h5py, time
from tqdm import tqdm

NPZ_ROOT = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/region_emb_extract-a100-0.65dedup"
DST_H5   = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/h5_embeddings/region_emb_flat.h5"
EMB_DIM  = 256

# ── 1. Scan filesystem (fast, ~30s) ──
print("1/3  Scanning filesystem...")
t0 = time.time()
classes = sorted(
    [d for d in os.listdir(NPZ_ROOT)
     if os.path.isdir(os.path.join(NPZ_ROOT, d)) and d.isdigit()],
    key=int,
)
sample_list = []  # (class_id_int, sample_name, npz_path)
for c in tqdm(classes, desc="Scanning classes"):
    mdir = os.path.join(NPZ_ROOT, c, "masks_npz")
    if not os.path.exists(mdir):
        continue
    for fn in sorted(os.listdir(mdir)):
        if fn.endswith(".npz"):
            sample_list.append((int(c), fn[:-4], os.path.join(mdir, fn)))
n_samples = len(sample_list)
print(f"     {n_samples:,} NPZ files, {len(classes)} classes  [{time.time()-t0:.1f}s]")

# Probe 200 files to estimate total segments
rng = np.random.default_rng(42)
probe = rng.choice(n_samples, min(200, n_samples), replace=False)
seg_counts = [np.load(sample_list[i][2], allow_pickle=True)["emb"].shape[0] for i in probe]
est_segs = int(np.mean(seg_counts) * n_samples * 1.05)
print(f"     Est. segments: {est_segs:,}  (avg {np.mean(seg_counts):.1f}/img)")

# ── 2. Write flat H5 (single pass over NPZs) ──
print(f"\n2/3  Writing {DST_H5} ...")
t1 = time.time()
os.makedirs(os.path.dirname(DST_H5), exist_ok=True)

with h5py.File(DST_H5, "w") as dst:
    # Flat arrays — resizable in case estimate is low
    emb_ds    = dst.create_dataset("emb",    shape=(est_segs, EMB_DIM), dtype="float32",
                                   maxshape=(None, EMB_DIM), chunks=(min(8192, est_segs), EMB_DIM))
    scores_ds = dst.create_dataset("scores", shape=(est_segs,), dtype="float32",
                                   maxshape=(None,), chunks=(min(65536, est_segs),))
    # Per-sample index (exact size)
    offsets_ds    = dst.create_dataset("offsets",     shape=(n_samples,), dtype="int64")
    n_seg_ds      = dst.create_dataset("n_segments",  shape=(n_samples,), dtype="int32")
    class_ids_ds  = dst.create_dataset("class_ids",   shape=(n_samples,), dtype="int32")
    names_ds      = dst.create_dataset("names",       shape=(n_samples,), dtype=h5py.string_dtype())
    mask_shapes_ds= dst.create_dataset("mask_shapes", shape=(n_samples, 3), dtype="int32")

    cursor = 0
    errs = []
    for si, (cid, name, path) in enumerate(tqdm(sample_list, desc="Writing H5")):
        try:
            d      = np.load(path, allow_pickle=True)
            emb    = d["emb"]       # (N, 256) float32
            sc     = d["scores"]    # (N,)
            shp    = d["shape"]     # (3,)  [N, H, W]
            n      = emb.shape[0]

            # Grow if needed
            if cursor + n > emb_ds.shape[0]:
                new = int(emb_ds.shape[0] * 1.5)
                emb_ds.resize(new, axis=0)
                scores_ds.resize(new, axis=0)

            emb_ds[cursor:cursor+n]    = emb
            scores_ds[cursor:cursor+n] = sc
            offsets_ds[si]    = cursor
            n_seg_ds[si]      = n
            class_ids_ds[si]  = cid
            names_ds[si]      = name
            mask_shapes_ds[si]= shp
            cursor += n
        except Exception as e:
            errs.append((path, str(e)))
            offsets_ds[si]    = cursor
            n_seg_ds[si]      = 0
            class_ids_ds[si]  = cid
            names_ds[si]      = name
            mask_shapes_ds[si]= [0,0,0]

    # Trim
    emb_ds.resize(cursor, axis=0)
    scores_ds.resize(cursor, axis=0)

    dst.attrs["total_samples"]  = n_samples
    dst.attrs["total_segments"] = cursor
    dst.attrs["emb_dim"]        = EMB_DIM
    dst.attrs["emb_dtype"]      = "float32"
    dst.attrs["source"]         = os.path.basename(NPZ_ROOT)

el = time.time()-t0
sz = os.path.getsize(DST_H5)
print(f"\n3/3  Done in {el:.0f}s ({el/60:.1f} min)")
print(f"     {cursor:,} segments, {n_samples:,} samples")
print(f"     File: {sz/(1024**3):.2f} GB")
if errs: print(f"     Errors: {len(errs)}  {errs[:3]}")

# Quick verify
print("\n     Verify (5 random):")
with h5py.File(DST_H5, "r") as f:
    for _ in range(5):
        i = rng.integers(0, n_samples)
        off, ns = int(f["offsets"][i]), int(f["n_segments"][i])
        cid = int(f["class_ids"][i])
        nm  = f["names"][i]; nm = nm.decode() if isinstance(nm, bytes) else nm
        h5e = f["emb"][off:off+ns]
        orig = np.load(os.path.join(NPZ_ROOT, str(cid), "masks_npz", f"{nm}.npz"),
                       allow_pickle=True)["emb"]
        ok = np.array_equal(h5e, orig)
        print(f"       idx={i} class={cid} name={nm} seg={ns} exact_match={ok}")

1/3  Scanning filesystem...


Scanning classes: 100%|██████████| 1000/1000 [00:01<00:00, 911.90it/s]


     510,172 NPZ files, 1000 classes  [1.2s]
     Est. segments: 22,611,078  (avg 42.2/img)

2/3  Writing /scratch/gilbreth/abelde/Thesis/StructureAwareGen/h5_embeddings/region_emb_flat.h5 ...


Writing H5:   0%|          | 1376/510172 [00:39<4:05:50, 34.49it/s] 


KeyboardInterrupt: 

In [2]:
# Paste this in a notebook cell and run.

import json
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor
import os

ROOT = Path("/scratch/gilbreth/abelde/Thesis/StructureAwareGen/region_emb_extract-a100-0.65dedup")
WORKERS = os.cpu_count() or 8
CHUNKSIZE = 256
PROGRESS_EVERY = 50_000  # set 0 to disable progress prints


def parse_one_json(path_str: str):
    # Returns tuple:
    # (has_num_masks, value, missing_key, bad_json, invalid_value)
    try:
        with open(path_str, "r", encoding="utf-8") as f:
            obj = json.load(f)
    except Exception:
        return 0, 0.0, 0, 1, 0

    if "num_masks" not in obj:
        return 0, 0.0, 1, 0, 0

    v = obj["num_masks"]
    if isinstance(v, (int, float)):
        return 1, float(v), 0, 0, 0

    try:
        return 1, float(v), 0, 0, 0
    except Exception:
        return 0, 0.0, 0, 0, 1


def iter_meta_jsons(root: Path):
    # Expected layout: root/<class_id>/meta/*.json
    for d in root.iterdir():
        if d.is_dir() and d.name.isdigit():
            meta_dir = d / "meta"
            if meta_dir.is_dir():
                for p in meta_dir.glob("*.json"):
                    yield str(p)


# ---------- Structure summary ----------
top_dirs = [p for p in ROOT.iterdir() if p.is_dir()]
numeric_dirs = [p for p in top_dirs if p.name.isdigit()]
non_numeric_dirs = [p.name for p in top_dirs if not p.name.isdigit()]
num_meta_dirs = sum((d / "meta").is_dir() for d in numeric_dirs)
num_masks_dirs = sum((d / "masks_npz").is_dir() for d in numeric_dirs)

print("=== Folder Structure ===")
print(f"root: {ROOT}")
print(f"top-level numeric class dirs: {len(numeric_dirs)}")
print(f"class dirs with meta/: {num_meta_dirs}")
print(f"class dirs with masks_npz/: {num_masks_dirs}")
if non_numeric_dirs:
    print(f"non-numeric top-level dirs: {len(non_numeric_dirs)}")
    print(f"examples: {non_numeric_dirs[:10]}")

# ---------- num_masks aggregation ----------
json_paths = list(iter_meta_jsons(ROOT))
total_json = len(json_paths)
print(f"\ntotal meta JSON files: {total_json}")

count_with_key = 0
sum_num_masks = 0.0
min_num_masks = None
max_num_masks = None
missing_key = 0
bad_json = 0
invalid_value = 0

with ProcessPoolExecutor(max_workers=WORKERS) as ex:
    for i, (has_key, val, miss, bad, inval) in enumerate(
        ex.map(parse_one_json, json_paths, chunksize=CHUNKSIZE), start=1
    ):
        if has_key:
            count_with_key += 1
            sum_num_masks += val
            if min_num_masks is None or val < min_num_masks:
                min_num_masks = val
            if max_num_masks is None or val > max_num_masks:
                max_num_masks = val

        missing_key += miss
        bad_json += bad
        invalid_value += inval

        if PROGRESS_EVERY and i % PROGRESS_EVERY == 0:
            print(f"processed {i}/{total_json}...")

avg_num_masks = (sum_num_masks / count_with_key) if count_with_key else float("nan")

print("\n=== num_masks Stats (all meta/*.json) ===")
print(f"files processed: {total_json}")
print(f"files with num_masks: {count_with_key}")
print(f"missing num_masks key: {missing_key}")
print(f"bad JSON files: {bad_json}")
print(f"invalid num_masks values: {invalid_value}")
print(f"sum(num_masks): {sum_num_masks:.0f}")
print(f"avg(num_masks): {avg_num_masks:.12f}")
print(f"min(num_masks): {min_num_masks}")
print(f"max(num_masks): {max_num_masks}")


=== Folder Structure ===
root: /scratch/gilbreth/abelde/Thesis/StructureAwareGen/region_emb_extract-a100-0.65dedup
top-level numeric class dirs: 1000
class dirs with meta/: 1000
class dirs with masks_npz/: 1000

total meta JSON files: 510172
processed 50000/510172...
processed 100000/510172...
processed 150000/510172...
processed 200000/510172...
processed 250000/510172...
processed 300000/510172...
processed 350000/510172...
processed 400000/510172...
processed 450000/510172...
processed 500000/510172...

=== num_masks Stats (all meta/*.json) ===
files processed: 510172
files with num_masks: 510172
missing num_masks key: 0
bad JSON files: 0
invalid num_masks values: 0
sum(num_masks): 19495720
avg(num_masks): 38.214014097206
min(num_masks): 0.0
max(num_masks): 100.0


In [3]:
import os
import re
import json
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

ROOT = Path("/scratch/gilbreth/abelde/Thesis/StructureAwareGen/region_emb_extract-a100-0.65dedup")
PROGRESS_EVERY = 50_000

num_masks_re = re.compile(r'"num_masks"\s*:\s*(\d+)')

def read_num_masks(path: Path) -> int:
    # Fast path: num_masks is near top of file, so scan first ~40 lines only.
    with path.open("r", encoding="utf-8") as f:
        for _ in range(40):
            line = f.readline()
            if not line:
                break
            m = num_masks_re.search(line)
            if m:
                return int(m.group(1))

    # Fallback (should rarely happen)
    with path.open("r", encoding="utf-8") as f:
        return int(json.load(f)["num_masks"])

def iter_meta_jsons(root: Path):
    for cls_dir in root.iterdir():
        if cls_dir.is_dir() and cls_dir.name.isdigit():
            meta_dir = cls_dir / "meta"
            if meta_dir.is_dir():
                yield from meta_dir.glob("*.json")

count = 0
total = 0
mn = None
mx = None

workers = min(64, (os.cpu_count() or 8) * 4)  # good for many small-file reads
with ThreadPoolExecutor(max_workers=workers) as ex:
    for i, n in enumerate(ex.map(read_num_masks, iter_meta_jsons(ROOT)), start=1):
        count += 1
        total += n
        mn = n if mn is None else min(mn, n)
        mx = n if mx is None else max(mx, n)

        if PROGRESS_EVERY and i % PROGRESS_EVERY == 0:
            print(f"processed {i} files...")

avg = total / count if count else float("nan")

print("\nDone")
print(f"files: {count}")
print(f"sum(num_masks): {total}")
print(f"avg(num_masks): {avg:.12f}")
print(f"min(num_masks): {mn}")
print(f"max(num_masks): {mx}")


processed 50000 files...
processed 100000 files...
processed 150000 files...
processed 200000 files...
processed 250000 files...
processed 300000 files...
processed 350000 files...
processed 400000 files...
processed 450000 files...
processed 500000 files...

Done
files: 510172
sum(num_masks): 19495720
avg(num_masks): 38.214014097206
min(num_masks): 0
max(num_masks): 100
